In [1]:
# Core libraries
import numpy as np
import pandas as pd

# Saving models
import joblib
import os

# NLP & ML
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

# File handling
from pathlib import Path
import json

In [2]:
def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)


def save_json(path, payload):
    ensure_dir(Path(path).parent)
    
    with open(path, "w") as f:
        json.dump(payload, f, indent=2)

In [3]:
def load_documents():
    return [
        # Tech
        "AI models are transforming software development and automation",
        "Machine learning enables intelligent systems",
        "Neural networks power modern applications",
        "Cloud computing supports scalable AI",

        # Finance
        "Stock markets react to interest rate changes",
        "Investment banking manages financial risk",
        "Economic growth depends on fiscal policy",
        "Money management improves financial stability",

        # Health
        "Doctors recommend regular medical checkups",
        "Healthy diets improve mental health",
        "Exercise improves cardiovascular fitness",
        "Hospitals use digital health systems",

        # Sports
        "Football teams prepare for championship season",
        "The player scored a winning goal",
        "Basketball is popular worldwide",
        "Athletes train daily for performance"
    ]


# Load documents
documents = load_documents()

print("Total documents:", len(documents))
documents

Total documents: 16


['AI models are transforming software development and automation',
 'Machine learning enables intelligent systems',
 'Neural networks power modern applications',
 'Cloud computing supports scalable AI',
 'Stock markets react to interest rate changes',
 'Investment banking manages financial risk',
 'Economic growth depends on fiscal policy',
 'Money management improves financial stability',
 'Doctors recommend regular medical checkups',
 'Healthy diets improve mental health',
 'Exercise improves cardiovascular fitness',
 'Hospitals use digital health systems',
 'Football teams prepare for championship season',
 'The player scored a winning goal',
 'Basketball is popular worldwide',
 'Athletes train daily for performance']

In [4]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=2000
)

X = vectorizer.fit_transform(documents)

print("TF-IDF Shape:", X.shape)

TF-IDF Shape: (16, 71)


In [5]:
n_topics = 4

nmf_model = NMF(
    n_components=n_topics,
    random_state=42
)

nmf_model.fit(X)

print("NMF model trained!")

NMF model trained!


In [6]:
ensure_dir("models")

joblib.dump(
    {
        "model": nmf_model,
        "vectorizer": vectorizer
    },
    "models/nmf_topic_model.joblib"
)

print("Model saved!")

Model saved!


In [7]:
feature_names = vectorizer.get_feature_names_out()

topics = []

for idx, topic in enumerate(nmf_model.components_):

    top_indices = topic.argsort()[-8:][::-1]

    top_words = [feature_names[i] for i in top_indices]

    topics.append({
        "Topic": idx,
        "Top Words": ", ".join(top_words)
    })

topics_df = pd.DataFrame(topics)

topics_df

,Topic,Top Words
0,0,"improves, financial, stability, money, managem..."
1,1,"systems, health, use, hospitals, digital, mach..."
2,2,"ai, supports, cloud, scalable, computing, deve..."
3,3,"train, athletes, performance, daily, season, t..."


In [8]:
ensure_dir("reports")

topics_df.to_csv("reports/topics.csv", index=False)

print("Topics saved to reports/topics.csv")

Topics saved to reports/topics.csv


In [9]:
save_json(
    "reports/metrics.json",
    {
        "model": "NMF",
        "n_topics": n_topics,
        "num_documents": len(documents)
    }
)

print("Metrics saved!")

Metrics saved!


In [10]:
# New unseen document
new_text = [
    "AI and cloud computing are improving business software"
]

# Convert text to TF-IDF
new_X = vectorizer.transform(new_text)

# Get topic distribution
topic_scores = nmf_model.transform(new_X)

# Display
for i, score in enumerate(topic_scores[0]):
    print(f"Topic {i}: {score:.4f}")

Topic 0: 0.0000
Topic 1: 0.0000
Topic 2: 0.6394
Topic 3: 0.0000
